# Classificador de Grãos de Soja — Treino YOLO11s (classificação)

**Dataset:** SoyaBeans Classifications (Roboflow / hansaka-sudusinghe) — MIT License — mesmo das ~12.500 imagens usado no EfficientNet  
**Modelo:** `yolo11s-cls.pt` (variante de **classificação** do YOLO11) — Transfer Learning (ImageNet → 5 classes)  
**Framework:** Ultralytics (PyTorch)  

### Por que `yolo11s-cls` e não o YOLO11s de detecção?
O dataset que temos é de **classificação** (pastas por classe, **sem** bounding boxes anotadas). Foi exatamente isso que fizemos no EfficientNet: 1 grão por imagem → 1 classe. O equivalente direto em YOLO é a variante `-cls`. O YOLO11s de **detecção** exigiria labels com caixas (`.txt` no formato YOLO), que este dataset não tem. A segmentação dos grãos continua sendo feita pelo OpenCV no `app.py`.

### Pré-requisitos
1. Dataset extraído no Google Drive na raiz ("Meu Drive"): `Meu Drive/SoyaBeans Classifications.v2i.folder (Unzipped Files)/`  
2. GPU ligada: `Ambiente de execução → Alterar tipo → T4 GPU`

### Classes (5) — ordem alfabética = índice do modelo
| Índice | Pasta no dataset | Rótulo curto |
|--------|------------------|--------------|
| 0 | Broken soybeans | broken |
| 1 | Immature soybeans | immature |
| 2 | Intact soybeans | intact |
| 3 | Skin-damaged soybeans | skin-damaged |
| 4 | Spotted soybeans | spotted |

> O folder `Part of the original soybean images` é **ignorado** (não é copiado para o dataset de treino), igual ao `class_names` explícito do EfficientNet.

In [ ]:
# Célula 1 — Instalar Ultralytics e verificar GPU
!pip -q install ultralytics

import torch
from ultralytics import YOLO
import ultralytics

print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA OK:     {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'Sem GPU! Vá em Ambiente de execução → Alterar tipo → T4 GPU'
print(f'GPU:         {torch.cuda.get_device_name(0)}')

In [ ]:
# Célula 2 — Montar Google Drive e configurar caminhos
import os
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
SAVE_DIR     = '/content/drive/MyDrive'

for split in ['train', 'valid', 'test']:
    p = os.path.join(DATASET_ROOT, split)
    assert os.path.isdir(p), f'Caminho não encontrado: {p}'
    print(f'OK: {p}')

# Classes exatas (ordem alfabética = índice do modelo)
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
print('\nClasses:', CLASS_NAMES)

In [ ]:
# Célula 3 — Preparar o dataset no formato de classificação do Ultralytics
# (com BALANCEAMENTO por oversampling no train)
#
# O YOLO-cls espera uma raiz com train/ val/ test/, cada um com 1 subpasta por
# classe. O export Roboflow usa 'valid' (e não 'val') e inclui a pasta extra
# 'Part of the original soybean images'. Copiamos para o disco LOCAL do Colab
# (treinar lendo do Drive é lento), renomeando valid -> val e copiando APENAS
# as 5 classes — a pasta 'Part of...' fica de fora.
#
# Balanceamento: o YOLO11-cls NÃO tem parâmetro de peso por classe (diferente do
# class_weight='balanced' do EfficientNet). O equivalente é OVERSAMPLING só no
# train: duplicamos imagens das classes menores até todas empatarem com a maior.
# val/ e test/ ficam INTOCADOS — nunca se balanceia conjunto de avaliação.
import shutil, glob

DST = '/content/soja_yolo'
SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}
IMG_EXTS = ('*.jpg', '*.jpeg', '*.png')

if os.path.exists(DST):
    shutil.rmtree(DST)

def list_imgs(d):
    files = []
    for e in IMG_EXTS:
        files.extend(glob.glob(os.path.join(d, e)))
    return sorted(files)

# 1) Cópia crua dos 3 splits (só as 5 classes)
counts = {}
for src_split, dst_split in SPLIT_MAP.items():
    counts[dst_split] = {}
    for cls in CLASS_NAMES:
        s = os.path.join(DATASET_ROOT, src_split, cls)
        d = os.path.join(DST, dst_split, cls)
        assert os.path.isdir(s), f'Faltando no dataset: {s}'
        shutil.copytree(s, d)
        counts[dst_split][SHORT[cls]] = len(list_imgs(d))

# 2) Oversampling SÓ no train: leva cada classe ao tamanho da maior.
#    Ciclo determinístico pelas originais (base[k % n]) → reprodutível e
#    distribui as cópias por igual em vez de duplicar sempre as mesmas.
train_counts = {cls: len(list_imgs(os.path.join(DST, 'train', cls))) for cls in CLASS_NAMES}
target = max(train_counts.values())
print('Antes do balanceamento (train):', {SHORT[c]: train_counts[c] for c in CLASS_NAMES})
print(f'Alvo por classe (= maior classe): {target}\n')

for cls in CLASS_NAMES:
    d = os.path.join(DST, 'train', cls)
    base = list_imgs(d)
    for k in range(target - len(base)):
        src = base[k % len(base)]
        stem, ext = os.path.splitext(os.path.basename(src))
        shutil.copy(src, os.path.join(d, f'{stem}__os{k:05d}{ext}'))
    counts['train'][SHORT[cls]] = len(list_imgs(d))

# 3) Resumo final
print('Dataset preparado em', DST, '(pasta "Part of..." excluída)\n')
for split, per_cls in counts.items():
    total = sum(per_cls.values())
    tag = '  (balanceado por oversampling)' if split == 'train' else ''
    print(f'{split:>5}: {total:>6} imagens  {per_cls}{tag}')
grand = sum(sum(c.values()) for c in counts.values())
print(f'\nTotal: {grand} imagens')

In [ ]:
# Célula 4 — Treinar YOLO11s-cls (GPU, batch 32, imgsz 224)
#
# imgsz=224 mantém paridade com o EfficientNet (mesmo resize). batch=32 conforme
# pedido. O Ultralytics já aplica augmentation própria (flip, rotação, HSV, etc.)
# e ordena as classes alfabeticamente — bate com CLASS_NAMES.
model = YOLO('yolo11s-cls.pt')

results = model.train(
    data=DST,
    epochs=100,
    imgsz=224,
    batch=32,
    device=0,
    patience=15,        # early stopping: para se não melhorar em 15 épocas
    seed=42,            # reprodutibilidade
    project='/content/runs_soja',
    name='yolo11s_cls',
    exist_ok=True,
    verbose=True,
)

print('\nMapa de classes do modelo (idx -> nome):')
print(model.names)
# Confirma que a ordem do modelo bate com a nossa lista de índices
assert [model.names[i] for i in range(len(CLASS_NAMES))] == CLASS_NAMES, \
    'Ordem das classes divergente — revisar antes de usar os índices!'
print('Ordem das classes confere com CLASS_NAMES.')

In [ ]:
# Célula 5 — Curvas de treino (o Ultralytics salva results.png automaticamente)
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

save_dir = str(model.trainer.save_dir)
print('Artefatos do treino em:', save_dir)

results_png = os.path.join(save_dir, 'results.png')
if os.path.exists(results_png):
    plt.figure(figsize=(14, 8))
    plt.imshow(mpimg.imread(results_png))
    plt.axis('off')
    plt.title('Curvas de treino — loss / acurácia (top1, top5)')
    plt.show()
else:
    print('results.png ainda não gerado.')

In [ ]:
# Célula 6 — Avaliação rápida no split de TESTE (acurácia top-1 / top-5)
metrics = model.val(split='test')
print(f'Top-1 (teste): {metrics.top1:.2%}')
print(f'Top-5 (teste): {metrics.top5:.2%}')

In [ ]:
# Célula 7 — Relatório de classificação + matriz de confusão (sklearn)
# Predizemos imagem a imagem no split de teste para montar y_true / y_pred.
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

idx_of = {name: i for i, name in enumerate(CLASS_NAMES)}

test_paths, y_true = [], []
for cls in CLASS_NAMES:
    for e in IMG_EXTS:
        for p in glob.glob(os.path.join(DST, 'test', cls, e)):
            test_paths.append(p)
            y_true.append(idx_of[cls])
print(f'Imagens de teste: {len(test_paths)}')

# stream=True evita acumular todos os Results na memória de uma vez
y_pred = []
for r in model.predict(test_paths, imgsz=224, batch=32, stream=True, verbose=False):
    y_pred.append(idx_of[model.names[int(r.probs.top1)]])

y_true = np.array(y_true)
y_pred = np.array(y_pred)
short_names = [SHORT[c] for c in CLASS_NAMES]

acc = (y_true == y_pred).mean()
print('\n=== Relatório de Classificação (teste) ===')
print(classification_report(y_true, y_pred, target_names=short_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Blues', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'Matriz de Confusão — YOLO11s-cls — Acurácia: {acc:.1%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/soja_yolo_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matriz salva no Drive.')

In [ ]:
# Célula 7b (CORRIGIDA) — Domain shift COM o mesmo recorte OpenCV do app.py
#
# Bug da versão anterior: eu mandava a foto crua (3 MB, grão pequeno num quadro
# grande) direto pro modelo, que treinou em recortes apertados. Deu 5.3% (abaixo
# do acaso) — entrada errada, não domain shift. O app SEMPRE recorta o grão
# (Otsu -> maior contorno -> bounding box) antes de classificar. Replicamos isso.
import cv2, tempfile, pathlib
import numpy as np
from PIL import Image
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download

def crop_single_grain(arr):
    """Mesmo recorte do app.py: maior contorno via Otsu -> bounding box (+pad)."""
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    ratio = cv2.contourArea(largest) / (h * w)
    if ratio <= 0.02 or ratio >= 0.95:
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

HF_TOKEN = getpass('Cole seu HF token (read): ')
CORR_REPO = 'Guguinhaxd/soja-correction'
api = HfApi(token=HF_TOKEN)
all_files = list(api.list_repo_files(CORR_REPO, repo_type='dataset'))
IMG_EXTS_SET = {'.jpg', '.jpeg', '.png'}
idx_of = {name: i for i, name in enumerate(CLASS_NAMES)}
tmpdir = tempfile.mkdtemp(prefix='corr_eval_')

crops, real_true = [], []
for cls in CLASS_NAMES:
    cls_files = [f for f in all_files
                 if f.startswith(f'{cls}/') and pathlib.Path(f).suffix.lower() in IMG_EXTS_SET]
    print(f'{SHORT[cls]:>15}: {len(cls_files)} fotos')
    for hf_path in cls_files:
        local = hf_hub_download(CORR_REPO, hf_path, repo_type='dataset',
                                token=HF_TOKEN, local_dir=tmpdir)
        arr = np.array(Image.open(local).convert('RGB'))
        crop = crop_single_grain(arr)          # <<< o passo que faltava
        crops.append(Image.fromarray(crop))    # PIL = RGB (sem confusão BGR)
        real_true.append(idx_of[cls])

print(f'\nTotal: {len(crops)} graos recortados')

# Prova visual: 5 recortes (confirma que o crop pegou o grão, nao o quadro inteiro)
plt.figure(figsize=(15, 3))
for i in range(min(5, len(crops))):
    plt.subplot(1, 5, i + 1)
    plt.imshow(crops[i]); plt.axis('off')
    plt.title(short_names[real_true[i]], fontsize=9)
plt.suptitle('Recortes enviados ao modelo (devem mostrar 1 grão enquadrado)')
plt.tight_layout(); plt.show()

real_pred = []
for r in model.predict(crops, imgsz=224, batch=32, stream=True, verbose=False):
    real_pred.append(idx_of[model.names[int(r.probs.top1)]])

real_true = np.array(real_true); real_pred = np.array(real_pred)
real_acc = (real_true == real_pred).mean()
print('\n=== Domain Shift (COM recorte, igual ao app) ===')
print(f'Acuracia YOLO11s-cls:  {real_acc:.1%}')
print(f'Acuracia EfficientNet: ~29%  (baseline pre-fine-tuning)')
delta = real_acc - 0.29
print(f'Delta vs EfficientNet: {"+" if delta >= 0 else ""}{delta:.1%}')

print('\n=== Relatorio por classe (fotos reais, recortadas) ===')
print(classification_report(real_true, real_pred, target_names=short_names, zero_division=0))

cm = confusion_matrix(real_true, real_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Oranges', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'Domain Shift (recortado) - YOLO11s-cls - {real_acc:.1%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/soja_yolo_domain_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matriz salva no Drive.')

In [ ]:
# Célula 8 — Salvar o melhor modelo + lista de classes no Drive
import json
from pathlib import Path

best_pt = Path(model.trainer.best)   # .../weights/best.pt
assert best_pt.exists(), f'best.pt não encontrado em {best_pt}'

DEST_PT      = f'{SAVE_DIR}/soja_yolo11s_best.pt'
CLASSES_PATH = f'{SAVE_DIR}/soja_classes.json'
shutil.copy(best_pt, DEST_PT)
with open(CLASSES_PATH, 'w') as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print(f'Modelo salvo:  {DEST_PT}')
print(f'Classes salvas: {CLASSES_PATH}')
print(f'Tamanho: {best_pt.stat().st_size / 1e6:.1f} MB')
print()
print('Próximos passos:')
print('1. Baixe soja_yolo11s_best.pt do Drive')
print('2. Compare a acurácia com a do EfficientNet (mesmo split de teste)')
print('3. Se for melhor, adaptamos o app.py para carregar o .pt do YOLO')

In [ ]:
# Célula 9 (OPCIONAL) — Inspecionar erros do modelo no teste
from PIL import Image

wrong_idx = np.where(y_pred != y_true)[0]
print(f'Total de erros: {len(wrong_idx)} de {len(y_true)} ({len(wrong_idx)/len(y_true):.1%})')

show = wrong_idx[:10]
plt.figure(figsize=(15, 6))
for i, idx in enumerate(show):
    plt.subplot(2, 5, i + 1)
    plt.imshow(Image.open(test_paths[idx]).convert('RGB'))
    plt.title(f'Real: {short_names[y_true[idx]]}\nPred: {short_names[y_pred[idx]]}',
              fontsize=8, color='red')
    plt.axis('off')
plt.suptitle('Exemplos de erros do YOLO11s-cls', fontsize=12)
plt.tight_layout()
plt.show()